In [0]:
%sql
select * from vg_sales.02_silver.fact_vg_sales_main

In [0]:
from pyspark.sql import functions as F

# 1. Carregar a Fato da camada Silver
# Esta tabela contém as métricas e os hashes (nossa ponte de ligação)
df_fato_silver = spark.table("vg_sales.02_silver.fact_vg_sales_main")

# 2. Carregar as Dimensões da camada Silver
# Filtramos por is_current = true para garantir que pegamos a versão ativa (SCD Tipo 2)
df_dim_console   = spark.table("vg_sales.02_silver.dim_console").filter("is_current = true")
df_dim_publisher = spark.table("vg_sales.02_silver.dim_publisher").filter("is_current = true")
df_dim_genre     = spark.table("vg_sales.02_silver.dim_genre").filter("is_current = true")

# 3. Realizar os Joins para substituir os Hashes pelas SKs (IDs numéricos)
# Usamos "left" join para garantir que não perdemos nenhuma linha da fato
df_gold_joined = df_fato_silver.join(
    df_dim_console, 
    df_fato_silver.console_hash == df_dim_console.console_hash, 
    "left"
).join(
    df_dim_publisher, 
    df_fato_silver.publisher_hash == df_dim_publisher.publisher_hash, 
    "left"
).join(
    df_dim_genre, 
    df_fato_silver.genre_hash == df_dim_genre.genre_hash, 
    "left"
)

display(df_gold_joined.limit(3))

In [0]:

# 4. Seleção final das colunas para a Gold
# Aqui removemos os hashes e nomes brutos da fato, deixando-a leve e performática
df_fato_gold_final = df_gold_joined.select(
    # Chaves Estrangeiras (SKs numéricas)
    F.col("console_sk"),
    F.col("publisher_sk"),
    F.col("genre_sk"),
    
    # Dados do Jogo e Métricas
    F.col("title"),
    F.col("developer"),
    F.col("critic_score"),
    F.col("total_sales"),
    F.col("na_sales"),
    F.col("pal_sales"),
    F.col("jp_sales"),
    F.col("other_sales"),
    F.col("img"),
    F.col("release_date"),
    F.col("last_update"),
    F.col("ingested_at"),
    
    F.col("release_year"),
    F.col("id_lote")
)
# 5. Carga Incremental com MERGE
# Criamos uma view temporária com os novos dados
df_fato_gold_final.createOrReplaceTempView("vw_stg_gold")

spark.sql("""
    MERGE INTO vg_sales.03_gold.fact_vg_sales AS target
    USING vw_stg_gold AS source
    ON target.id_lote = source.id_lote 
       AND target.release_year = source.release_year
    WHEN NOT MATCHED THEN
      INSERT *
""")

print("✅ Camada Gold atualizada com sucesso de forma incremental e particionada!")

In [0]:
df_dim_console_gold   = spark.table("vg_sales.02_silver.dim_console").filter("is_current = true").select("console_sk", "console_name")
df_dim_publisher_gold = spark.table("vg_sales.02_silver.dim_publisher").filter("is_current = true").select("publisher_sk", "publisher_name")
df_dim_genre_gold     = spark.table("vg_sales.02_silver.dim_genre").filter("is_current = true").select("genre_sk", "genre_name")


# 2. Salvando as Dimensões na Gold (Overwrite para manter o espelho atualizado)
df_dim_console_gold.write.format("delta").mode("overwrite").saveAsTable("vg_sales.03_gold.dim_console")
df_dim_publisher_gold.write.format("delta").mode("overwrite").saveAsTable("vg_sales.03_gold.dim_publisher")
df_dim_genre_gold.write.format("delta").mode("overwrite").saveAsTable("vg_sales.03_gold.dim_genre")

In [0]:
%sql
select * from vg_sales.03_gold.dim_genre